# LeetCode 661 — Image Smoother bằng Python

**Link đề:** https://leetcode.com/problems/image-smoother/
**Độ khó:** Easy · **Chủ đề:** Array, Matrix

```python
class Solution:
    def imageSmoother(self, img: list[list[int]]) -> list[list[int]]:
        ...
```

## Mục tiêu học
1. Làm quen với **ma trận 2D** trong Python: list lồng list, chỉ số `img[r][c]`, số hàng/cột.
2. Duyệt **8 ô láng giềng** quanh một ô bằng hai vòng `dr, dc ∈ {-1, 0, 1}` và **kiểm tra biên**.
3. Hiểu vì sao **không được sửa trực tiếp** ảnh gốc khi đang tính (cần ma trận kết quả riêng).
4. Biết cách **tổng tiền tố 2D** (2D prefix sum) để tính tổng hình chữ nhật trong `O(1)`.
5. Biết mẹo **mã hóa bit** để làm tại chỗ với `O(1)` bộ nhớ thêm.

**Cách học:** đọc markdown → đoán output → chạy cell → so sánh. Cell TODO không tự chạy.

## 1. Đề bài

Bộ làm mịn ảnh (image smoother) là bộ lọc `3 × 3`: giá trị mới của mỗi ô = **làm tròn xuống trung bình** của ô đó và **8 ô xung quanh**. Ô xung quanh nào **nằm ngoài ảnh** thì **không tính** (cả tổng lẫn số lượng).

Cho ma trận `img` kích thước `m × n` (ảnh xám), trả về ảnh sau khi làm mịn **mọi ô**.

**Ví dụ 1:** `img = [[1,1,1],[1,0,1],[1,1,1]]` → `[[0,0,0],[0,0,0],[0,0,0]]`
- Góc: `floor(3/4) = 0`; cạnh: `floor(5/6) = 0`; giữa: `floor(8/9) = 0`.

**Ví dụ 2:** `img = [[100,200,100],[200,50,200],[100,200,100]]` → `[[137,141,137],[141,138,141],[137,141,137]]`
- Góc: `floor((100+200+200+50)/4) = floor(137.5) = 137`.
- Cạnh: `floor((200+200+50+200+100+100)/6) = 141`.
- Giữa: `floor(1250/9) = 138`.

**Constraints:** `1 <= m, n <= 200`, `0 <= img[i][j] <= 255`.

Số ô được tính: **góc 4**, **cạnh 6**, **bên trong 9** (với ảnh ≥ 2×2). Ảnh `1 × 1` thì chỉ có 1 ô.

## 2. Nền tảng Python cần biết

### 2.1. Ma trận = list các list
`img[r]` là **hàng** `r` (một list), `img[r][c]` là ô ở hàng `r`, cột `c`.
Số hàng `m = len(img)`, số cột `n = len(img[0])`.

In [1]:
img = [[100, 200, 100],
       [200, 50, 200],
       [100, 200, 100]]
m, n = len(img), len(img[0])
print("m =", m, "| n =", n)
print("hàng 1:", img[1])
print("ô (1, 1):", img[1][1], "| ô (0, 2):", img[0][2])
for row in img:
    print(" ".join(f"{v:>4}" for v in row))

m = 3 | n = 3
hàng 1: [200, 50, 200]
ô (1, 1): 50 | ô (0, 2): 100
 100  200  100
 200   50  200
 100  200  100


### 2.2. Tạo ma trận kết quả: bẫy `[[0] * n] * m`
`[[0] * n] * m` tạo `m` **tham chiếu tới cùng một hàng** → sửa một ô thì cả cột đổi theo!
Cách đúng: list comprehension `[[0] * n for _ in range(m)]` tạo `m` hàng **độc lập**.

In [2]:
bad = [[0] * 3] * 2
bad[0][0] = 9
print("tạo sai :", bad)          # [[9,0,0],[9,0,0]] -> cả hai hàng bị đổi!

good = [[0] * 3 for _ in range(2)]
good[0][0] = 9
print("tạo đúng:", good)

tạo sai : [[9, 0, 0], [9, 0, 0]]
tạo đúng: [[9, 0, 0], [0, 0, 0]]


### 2.3. Duyệt 8 láng giềng với `dr, dc`
Với ô `(r, c)`, các ô trong khối 3×3 là `(r + dr, c + dc)` với `dr, dc ∈ {-1, 0, 1}` (9 tổ hợp, gồm chính nó khi `dr = dc = 0`).

In [3]:
r, c = 1, 1
for dr in (-1, 0, 1):
    print([(r + dr, c + dc) for dc in (-1, 0, 1)])

[(0, 0), (0, 1), (0, 2)]
[(1, 0), (1, 1), (1, 2)]
[(2, 0), (2, 1), (2, 2)]


### 2.4. Kiểm tra biên `0 <= nr < m and 0 <= nc < n`
Python cho phép viết **so sánh chuỗi** `0 <= x < m` (tương đương `0 <= x and x < m`).
⚠ Nếu quên kiểm tra, chỉ số `-1` **không báo lỗi** mà đọc phần tử cuối → sai âm thầm!

In [4]:
m, n = 3, 3
for (nr, nc) in [(0, 0), (-1, 0), (2, 3), (1, 2)]:
    print((nr, nc), "trong ảnh?" , 0 <= nr < m and 0 <= nc < n)

row = [100, 200, 100]
print("row[-1] =", row[-1], "-> Python không báo lỗi với chỉ số âm!")

(0, 0) trong ảnh? True
(-1, 0) trong ảnh? False
(2, 3) trong ảnh? False
(1, 2) trong ảnh? True
row[-1] = 100 -> Python không báo lỗi với chỉ số âm!


### 2.5. Làm tròn xuống: `//` và `math.floor`
Giá trị pixel không âm nên `total // count` chính là `floor(total / count)`.
(Với số âm, `//` vẫn làm tròn **xuống**, khác với `int()` làm tròn **về 0** — ở bài này không gặp số âm.)

In [5]:
import math
print(550 // 4, math.floor(550 / 4), int(550 / 4))
print(-7 // 2, math.floor(-7 / 2), int(-7 / 2))

137 137 137
-4 -4 -3


## 3. Cách trực giác — duyệt khối 3×3 cho từng ô

Với mỗi ô `(r, c)`:
1. `total = 0`, `count = 0`.
2. Duyệt 9 ô `(r+dr, c+dc)`; ô nào **nằm trong ảnh** thì cộng vào `total`, tăng `count`.
3. `res[r][c] = total // count`.

Mỗi ô tốn tối đa 9 phép → tổng `O(9·m·n) = O(m·n)`. Với `m, n ≤ 200` đây đã đủ nhanh — đây chính là lời giải chuẩn của bài.

In [6]:
def smoother_direct(img: list[list[int]]) -> list[list[int]]:
    m, n = len(img), len(img[0])
    res = [[0] * n for _ in range(m)]
    for r in range(m):
        for c in range(n):
            total = count = 0
            for dr in (-1, 0, 1):
                for dc in (-1, 0, 1):
                    nr, nc = r + dr, c + dc
                    if 0 <= nr < m and 0 <= nc < n:
                        total += img[nr][nc]
                        count += 1
            res[r][c] = total // count
    return res

print(smoother_direct([[1, 1, 1], [1, 0, 1], [1, 1, 1]]))
print(smoother_direct([[100, 200, 100], [200, 50, 200], [100, 200, 100]]))

[[0, 0, 0], [0, 0, 0], [0, 0, 0]]
[[137, 141, 137], [141, 138, 141], [137, 141, 137]]


## 4. Nút thắt và insight

**Nút thắt 1 — ghi đè:** nếu ghi kết quả thẳng vào `img`, các ô tính **sau** sẽ đọc phải giá trị **đã làm mịn** của ô trước → sai. Vì vậy cần ma trận `res` riêng (hoặc mẹo mã hóa bit ở mục 7).

**Nút thắt 2 — bộ lọc lớn:** nếu kích thước bộ lọc là `(2k+1)×(2k+1)` thay vì 3×3, cách trực tiếp tốn `O(m·n·k²)`. Khi đó dùng **tổng tiền tố 2D**: tổng một hình chữ nhật bất kỳ trong `O(1)`.

```text
P[i][j] = tổng các ô img[0..i-1][0..j-1]
Tổng hình chữ nhật hàng r1..r2, cột c1..c2
  = P[r2+1][c2+1] - P[r1][c2+1] - P[r2+1][c1] + P[r1][c1]
```
Số ô trong hình chữ nhật = `(r2 - r1 + 1) * (c2 - c1 + 1)` — tự động đúng ở biên nếu ta **kẹp** `r1 = max(0, r-1)`, `r2 = min(m-1, r+1)`, ...

In [7]:
img = [[100, 200, 100], [200, 50, 200], [100, 200, 100]]
r, c = 0, 0
r1, r2 = max(0, r - 1), min(2, r + 1)
c1, c2 = max(0, c - 1), min(2, c + 1)
print("ô (0,0): hàng", r1, "..", r2, "| cột", c1, "..", c2)
cells = [img[i][j] for i in range(r1, r2 + 1) for j in range(c1, c2 + 1)]
print("các ô:", cells, "-> tổng", sum(cells), "| số ô", len(cells), "| kết quả", sum(cells) // len(cells))

ô (0,0): hàng 0 .. 1 | cột 0 .. 1
các ô: [100, 200, 200, 50] -> tổng 550 | số ô 4 | kết quả 137


## 5. Lời giải chính — giảng từng đoạn

**Đoạn 1:** lấy kích thước `m, n`, tạo `res` toàn 0 (độc lập từng hàng).

**Đoạn 2:** hai vòng `for r`, `for c` duyệt mọi ô.

**Đoạn 3:** với mỗi ô, hai vòng `dr, dc` duyệt khối 3×3; **kiểm tra biên** trước khi đọc `img[nr][nc]`.

**Đoạn 4:** `res[r][c] = total // count`. `count` luôn ≥ 1 vì chính ô `(r, c)` luôn hợp lệ → không chia cho 0.

Phiên bản dưới viết gọn khối 3×3 bằng `range(max(0, r-1), min(m, r+2))` — tự kẹp biên, không cần `if`.

In [8]:
def smoother_clamped(img: list[list[int]]) -> list[list[int]]:
    m, n = len(img), len(img[0])                              # đoạn 1
    res = [[0] * n for _ in range(m)]
    for r in range(m):                                        # đoạn 2
        for c in range(n):
            total = count = 0
            for nr in range(max(0, r - 1), min(m, r + 2)):    # đoạn 3 (đã kẹp biên)
                for nc in range(max(0, c - 1), min(n, c + 2)):
                    total += img[nr][nc]
                    count += 1
            res[r][c] = total // count                        # đoạn 4
    return res

print(smoother_clamped([[100, 200, 100], [200, 50, 200], [100, 200, 100]]))
print(smoother_clamped([[7]]))
print(smoother_clamped([[1, 2, 3, 4]]))

[[137, 141, 137], [141, 138, 141], [137, 141, 137]]
[[7]]
[[1, 2, 3, 3]]


## 6. TRACE từng bước

Ảnh `[[100,200,100],[200,50,200],[100,200,100]]`. Trace 3 ô đại diện:

| Ô | Hàng hợp lệ | Cột hợp lệ | Các giá trị | `total` | `count` | `total // count` |
|---|---|---|---|---:|---:|---:|
| (0,0) góc | 0..1 | 0..1 | 100,200,200,50 | 550 | 4 | 137 |
| (0,1) cạnh | 0..1 | 0..2 | 100,200,100,200,50,200 | 850 | 6 | 141 |
| (1,1) giữa | 0..2 | 0..2 | cả 9 ô | 1250 | 9 | 138 |

In [9]:
def trace_cell(img: list[list[int]], r: int, c: int) -> int:
    m, n = len(img), len(img[0])
    total = count = 0
    print(f"Ô ({r},{c}) giá trị gốc {img[r][c]}:")
    for dr in (-1, 0, 1):
        for dc in (-1, 0, 1):
            nr, nc = r + dr, c + dc
            if 0 <= nr < m and 0 <= nc < n:
                total += img[nr][nc]
                count += 1
                print(f"   ({nr},{nc}) = {img[nr][nc]:>3} -> total={total}, count={count}")
            else:
                print(f"   ({nr},{nc}) ngoài ảnh -> bỏ qua")
    print(f"   kết quả = {total} // {count} = {total // count}")
    return total // count

img2 = [[100, 200, 100], [200, 50, 200], [100, 200, 100]]
trace_cell(img2, 0, 0)

Ô (0,0) giá trị gốc 100:
   (-1,-1) ngoài ảnh -> bỏ qua
   (-1,0) ngoài ảnh -> bỏ qua
   (-1,1) ngoài ảnh -> bỏ qua
   (0,-1) ngoài ảnh -> bỏ qua
   (0,0) = 100 -> total=100, count=1
   (0,1) = 200 -> total=300, count=2
   (1,-1) ngoài ảnh -> bỏ qua
   (1,0) = 200 -> total=500, count=3
   (1,1) =  50 -> total=550, count=4
   kết quả = 550 // 4 = 137


137

In [10]:
trace_cell(img2, 0, 1)
trace_cell(img2, 1, 1)

Ô (0,1) giá trị gốc 200:
   (-1,0) ngoài ảnh -> bỏ qua
   (-1,1) ngoài ảnh -> bỏ qua
   (-1,2) ngoài ảnh -> bỏ qua
   (0,0) = 100 -> total=100, count=1
   (0,1) = 200 -> total=300, count=2
   (0,2) = 100 -> total=400, count=3
   (1,0) = 200 -> total=600, count=4
   (1,1) =  50 -> total=650, count=5
   (1,2) = 200 -> total=850, count=6
   kết quả = 850 // 6 = 141
Ô (1,1) giá trị gốc 50:
   (0,0) = 100 -> total=100, count=1
   (0,1) = 200 -> total=300, count=2
   (0,2) = 100 -> total=400, count=3
   (1,0) = 200 -> total=600, count=4
   (1,1) =  50 -> total=650, count=5
   (1,2) = 200 -> total=850, count=6
   (2,0) = 100 -> total=950, count=7
   (2,1) = 200 -> total=1150, count=8
   (2,2) = 100 -> total=1250, count=9
   kết quả = 1250 // 9 = 138


138

## 7. Code nộp LeetCode

### 7.1. Cách 1 — Duyệt trực tiếp 3×3 (khuyên dùng)

In [11]:
class Solution:
    def imageSmoother(self, img: list[list[int]]) -> list[list[int]]:
        m, n = len(img), len(img[0])
        res = [[0] * n for _ in range(m)]
        for r in range(m):
            for c in range(n):
                total = count = 0
                for nr in range(max(0, r - 1), min(m, r + 2)):
                    for nc in range(max(0, c - 1), min(n, c + 2)):
                        total += img[nr][nc]
                        count += 1
                res[r][c] = total // count
        return res

print(Solution().imageSmoother([[1, 1, 1], [1, 0, 1], [1, 1, 1]]))
print(Solution().imageSmoother([[100, 200, 100], [200, 50, 200], [100, 200, 100]]))

[[0, 0, 0], [0, 0, 0], [0, 0, 0]]
[[137, 141, 137], [141, 138, 141], [137, 141, 137]]


### 7.2. Cách 2 — Tổng tiền tố 2D

Dựng `P` kích thước `(m+1) × (n+1)` với công thức:
```text
P[i+1][j+1] = img[i][j] + P[i][j+1] + P[i+1][j] - P[i][j]
```
(Cộng hình chữ nhật phía trên và bên trái, trừ phần giao bị cộng hai lần — **nguyên lý bao hàm–loại trừ**.)

In [12]:
def build_prefix_2d(img):
    m, n = len(img), len(img[0])
    P = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(m):
        for j in range(n):
            P[i + 1][j + 1] = img[i][j] + P[i][j + 1] + P[i + 1][j] - P[i][j]
    return P

P = build_prefix_2d([[1, 2, 3], [4, 5, 6], [7, 8, 9]])
for row in P:
    print(row)
# tổng hàng 1..2, cột 1..2 = 5+6+8+9 = 28
print("tổng khối (1..2, 1..2) =", P[3][3] - P[1][3] - P[3][1] + P[1][1])

[0, 0, 0, 0]
[0, 1, 3, 6]
[0, 5, 12, 21]
[0, 12, 27, 45]
tổng khối (1..2, 1..2) = 28


In [13]:
class SolutionPrefix:
    def imageSmoother(self, img: list[list[int]]) -> list[list[int]]:
        m, n = len(img), len(img[0])
        P = [[0] * (n + 1) for _ in range(m + 1)]
        for i in range(m):
            for j in range(n):
                P[i + 1][j + 1] = img[i][j] + P[i][j + 1] + P[i + 1][j] - P[i][j]
        res = [[0] * n for _ in range(m)]
        for r in range(m):
            r1, r2 = max(0, r - 1), min(m - 1, r + 1)
            for c in range(n):
                c1, c2 = max(0, c - 1), min(n - 1, c + 1)
                total = P[r2 + 1][c2 + 1] - P[r1][c2 + 1] - P[r2 + 1][c1] + P[r1][c1]
                count = (r2 - r1 + 1) * (c2 - c1 + 1)
                res[r][c] = total // count
        return res

print(SolutionPrefix().imageSmoother([[100, 200, 100], [200, 50, 200], [100, 200, 100]]))

[[137, 141, 137], [141, 138, 141], [137, 141, 137]]


### 7.3. Cách 3 — Làm tại chỗ bằng mã hóa bit (O(1) bộ nhớ thêm)

Pixel nằm trong `0..255` → chỉ cần **8 bit thấp**. Ta lưu **giá trị mới** vào các bit cao: `img[r][c] |= new << 8`.
- Khi đọc **giá trị gốc** của láng giềng: `img[nr][nc] & 255` (lấy 8 bit thấp) — không bị ảnh hưởng bởi kết quả đã ghi.
- Cuối cùng dịch phải: `img[r][c] >>= 8` để lấy giá trị mới.

Toán tử bit: `&` (AND), `|` (OR), `<<` dịch trái (nhân `2^k`), `>>` dịch phải (chia nguyên `2^k`).

In [14]:
v = 50
new = 138
packed = v | (new << 8)
print("packed =", packed, "| bin =", bin(packed))
print("gốc  = packed & 255 =", packed & 255)
print("mới  = packed >> 8  =", packed >> 8)

packed = 35378 | bin = 0b1000101000110010
gốc  = packed & 255 = 50
mới  = packed >> 8  = 138


In [15]:
class SolutionInPlace:
    def imageSmoother(self, img: list[list[int]]) -> list[list[int]]:
        m, n = len(img), len(img[0])
        for r in range(m):
            for c in range(n):
                total = count = 0
                for nr in range(max(0, r - 1), min(m, r + 2)):
                    for nc in range(max(0, c - 1), min(n, c + 2)):
                        total += img[nr][nc] & 255      # chỉ đọc giá trị gốc
                        count += 1
                img[r][c] |= (total // count) << 8      # cất kết quả vào bit cao
        for r in range(m):
            for c in range(n):
                img[r][c] >>= 8
        return img

print(SolutionInPlace().imageSmoother([[100, 200, 100], [200, 50, 200], [100, 200, 100]]))

[[137, 141, 137], [141, 138, 141], [137, 141, 137]]


## 8. Vì sao đúng?

**Cách trực tiếp:** với mỗi ô, tập `{(nr, nc)}` được duyệt chính là giao của khối 3×3 tâm `(r, c)` với ảnh — đúng định nghĩa "bỏ qua ô không tồn tại". `total` là tổng, `count` là số ô của tập đó, `//` là làm tròn xuống với số không âm. Vì chỉ **đọc** `img` và **ghi** `res`, mọi ô đều được tính từ ảnh gốc.

**Prefix 2D — bất biến:** `P[i][j]` = tổng hình chữ nhật `img[0..i-1][0..j-1]`. Hình chữ nhật `[r1..r2]×[c1..c2]` = hình lớn `P[r2+1][c2+1]` trừ dải trên `P[r1][c2+1]`, trừ dải trái `P[r2+1][c1]`, cộng lại góc bị trừ hai lần `P[r1][c1]`.

**Mã hóa bit:** bất biến — 8 bit thấp của mọi ô **luôn là giá trị gốc** trong suốt pha 1 (vì `new << 8` không chạm 8 bit thấp, và `new ≤ 255` nên không tràn sang nhau).

## 9. Độ phức tạp và so sánh

Gọi `m × n` là kích thước ảnh, `K` là cạnh bộ lọc (ở đây `K = 3`).

| Cách | Time | Space thêm | Ghi chú |
|---|---:|---:|---|
| Trực tiếp 3×3 | `O(m·n·K²)` = `O(9mn)` = `O(mn)` | `O(mn)` cho `res` (là output) | Đơn giản nhất |
| Prefix 2D | `O(mn)` | `O(mn)` cho `P` | Không phụ thuộc `K` |
| Mã hóa bit tại chỗ | `O(mn)` | `O(1)` | Sửa input |

Với `K = 3` cố định, cả ba đều `O(mn)`; prefix 2D chỉ thật sự hơn khi `K` lớn.

## 10. Bộ test

1. Hai ví dụ đề.
2. Biên: `1×1`, một hàng, một cột, `2×2`.
3. Ảnh toàn 255 (trung bình giữ nguyên 255).
4. Ngẫu nhiên: so 3 cách với nhau (nhớ **deepcopy** cho cách tại chỗ).

In [16]:
import copy, random

cases = [
    ([[1, 1, 1], [1, 0, 1], [1, 1, 1]], [[0, 0, 0], [0, 0, 0], [0, 0, 0]]),
    ([[100, 200, 100], [200, 50, 200], [100, 200, 100]], [[137, 141, 137], [141, 138, 141], [137, 141, 137]]),
    ([[7]], [[7]]),
    ([[1, 2, 3, 4]], [[1, 2, 3, 3]]),
    ([[1], [2], [3]], [[1], [2], [2]]),
    ([[0, 3], [5, 255]], [[65, 65], [65, 65]]),
    ([[255] * 3 for _ in range(3)], [[255] * 3 for _ in range(3)]),
]
solvers = [Solution().imageSmoother, SolutionPrefix().imageSmoother, SolutionInPlace().imageSmoother, smoother_direct]
for img, expected in cases:
    for f in solvers:
        got = f(copy.deepcopy(img))
        assert got == expected, (f, img, got, expected)
    print(f"{str(img):<45} PASS")

[[1, 1, 1], [1, 0, 1], [1, 1, 1]]             PASS
[[100, 200, 100], [200, 50, 200], [100, 200, 100]] PASS
[[7]]                                         PASS
[[1, 2, 3, 4]]                                PASS
[[1], [2], [3]]                               PASS
[[0, 3], [5, 255]]                            PASS
[[255, 255, 255], [255, 255, 255], [255, 255, 255]] PASS


In [17]:
rng = random.Random(661)
for _ in range(200):
    m, n = rng.randint(1, 7), rng.randint(1, 7)
    img = [[rng.randint(0, 255) for _ in range(n)] for _ in range(m)]
    exp = smoother_direct(img)
    for f in solvers:
        assert f(copy.deepcopy(img)) == exp
print("200 ảnh ngẫu nhiên: 3 cách khớp nhau PASS")

200 ảnh ngẫu nhiên: 3 cách khớp nhau PASS


## 11. Lỗi người mới hay gặp

### Lỗi 1: Ghi đè thẳng vào `img`

In [18]:
def wrong_overwrite(img):
    m, n = len(img), len(img[0])
    for r in range(m):
        for c in range(n):
            total = count = 0
            for nr in range(max(0, r - 1), min(m, r + 2)):
                for nc in range(max(0, c - 1), min(n, c + 2)):
                    total += img[nr][nc]
                    count += 1
            img[r][c] = total // count          # SAI: các ô sau đọc giá trị đã bị sửa
    return img

src = [[100, 200, 100], [200, 50, 200], [100, 200, 100]]
print("sai :", wrong_overwrite(copy.deepcopy(src)))
print("đúng:", Solution().imageSmoother(src))

sai : [[137, 147, 124], [139, 133, 150], [143, 144, 131]]
đúng: [[137, 141, 137], [141, 138, 141], [137, 141, 137]]


### Lỗi 2: Chia luôn cho 9
Ô góc/cạnh có ít láng giềng hơn → phải chia cho `count` thực tế.

### Lỗi 3: Quên kiểm tra biên
Chỉ số `-1` không lỗi mà đọc ô ở **đầu kia** của hàng/cột; chỉ số `m` thì `IndexError`.

In [19]:
def wrong_divide_9(img):
    m, n = len(img), len(img[0])
    res = [[0] * n for _ in range(m)]
    for r in range(m):
        for c in range(n):
            total = sum(img[nr][nc] for nr in range(max(0, r - 1), min(m, r + 2))
                                    for nc in range(max(0, c - 1), min(n, c + 2)))
            res[r][c] = total // 9            # SAI
    return res

print("chia 9 (sai):", wrong_divide_9(src)[0])

def wrong_no_bounds(img):
    m, n = len(img), len(img[0])
    r, c = 0, 0
    total = sum(img[r + dr][c + dc] for dr in (-1, 0, 1) for dc in (-1, 0, 1))
    return total // 9

print("không kiểm biên ở (0,0): đọc nhầm ô đối diện ->", wrong_no_bounds(src), "| đúng:", Solution().imageSmoother(src)[0][0])

chia 9 (sai): [61, 94, 61]
không kiểm biên ở (0,0): đọc nhầm ô đối diện -> 138 | đúng: 137


### Lỗi 4: `[[0] * n] * m`
Như mục 2.2: mọi hàng của `res` là **một** list → kết quả cuối mọi hàng giống hệt hàng cuối cùng.

### Lỗi 5: Nhầm `m` và `n`
`len(img)` là số **hàng**, `len(img[0])` là số **cột**. Test với ảnh **không vuông** (`1×4`, `3×1`) để bắt lỗi này.

In [20]:
def wrong_shared_rows(img):
    m, n = len(img), len(img[0])
    res = [[0] * n] * m                         # SAI
    for r in range(m):
        for c in range(n):
            vals = [img[nr][nc] for nr in range(max(0, r - 1), min(m, r + 2))
                                for nc in range(max(0, c - 1), min(n, c + 2))]
            res[r][c] = sum(vals) // len(vals)
    return res

print("sai :", wrong_shared_rows(src))
print("đúng:", Solution().imageSmoother(src))

sai : [[137, 141, 137], [137, 141, 137], [137, 141, 137]]
đúng: [[137, 141, 137], [141, 138, 141], [137, 141, 137]]


## 12. Mẹo nhận dạng pattern

- **"Mỗi ô phụ thuộc vào các ô xung quanh"** trên lưới → vòng `dr, dc` + kiểm tra biên. Có thể khai báo sẵn `DIRS = [(-1,-1), (-1,0), ...]`.
- **"Cập nhật đồng thời mọi ô"** (giá trị mới dựa trên giá trị **cũ**) → cần ma trận phụ hoặc mã hóa trạng thái cũ/mới trong cùng ô. Ví dụ kinh điển: LeetCode 289 Game of Life.
- **"Tổng hình chữ nhật con, nhiều truy vấn"** → prefix sum 2D (LeetCode 304, 1314 Matrix Block Sum).

## 13. Biến thể

### 13.1. Bộ lọc kích thước tùy ý `(2k+1)×(2k+1)` — Matrix Block Sum (LeetCode 1314)
Trả **tổng** (không chia) các ô trong bán kính `k`. Prefix 2D tỏa sáng ở đây.

In [21]:
def matrix_block_sum(mat, k):
    m, n = len(mat), len(mat[0])
    P = build_prefix_2d(mat)
    res = [[0] * n for _ in range(m)]
    for r in range(m):
        r1, r2 = max(0, r - k), min(m - 1, r + k)
        for c in range(n):
            c1, c2 = max(0, c - k), min(n - 1, c + k)
            res[r][c] = P[r2 + 1][c2 + 1] - P[r1][c2 + 1] - P[r2 + 1][c1] + P[r1][c1]
    return res

print(matrix_block_sum([[1, 2, 3], [4, 5, 6], [7, 8, 9]], 1))   # [[12,21,16],[27,45,33],[24,39,28]]

[[12, 21, 16], [27, 45, 33], [24, 39, 28]]


### 13.2. Đếm láng giềng sống — bước đầu của Game of Life

In [22]:
def live_neighbors(board, r, c):
    m, n = len(board), len(board[0])
    cnt = 0
    for dr in (-1, 0, 1):
        for dc in (-1, 0, 1):
            if dr == 0 and dc == 0:
                continue                      # không đếm chính nó
            nr, nc = r + dr, c + dc
            if 0 <= nr < m and 0 <= nc < n:
                cnt += board[nr][nc]
    return cnt

board = [[0, 1, 0], [0, 0, 1], [1, 1, 1], [0, 0, 0]]
print([[live_neighbors(board, r, c) for c in range(3)] for r in range(4)])

[[1, 1, 2], [3, 5, 3], [1, 3, 2], [2, 3, 2]]


## 14. Bài tập (tự làm trước, đáp án ở cuối)

**Bài 1 — Tính tay:** với `img = [[2, 4], [6, 8]]`, kết quả làm mịn là gì?

**Bài 2 — Đoán số ô:** trong ảnh `4 × 5`, có bao nhiêu ô được tính với `count = 4`, `count = 6`, `count = 9`?

**Bài 3 — TODO:** viết hàm `exercise_count_matrix(m, n)` trả về ma trận `count` (số ô hợp lệ quanh mỗi ô).

In [23]:
def exercise_count_matrix(m: int, n: int) -> list[list[int]]:
    # TODO: với mỗi (r, c), đếm số (nr, nc) trong khối 3x3 nằm trong ảnh
    pass

# assert exercise_count_matrix(3, 3) == [[4, 6, 4], [6, 9, 6], [4, 6, 4]]
# assert exercise_count_matrix(1, 1) == [[1]]

**Bài 4 — TODO:** viết bộ lọc **max 3×3** (mỗi ô = giá trị lớn nhất trong khối 3×3 hợp lệ) — kiểu "dilation" trong xử lý ảnh.

In [24]:
def exercise_max_filter(img: list[list[int]]) -> list[list[int]]:
    # TODO: giống imageSmoother nhưng thay tổng/đếm bằng max
    pass

# assert exercise_max_filter([[1, 2], [3, 4]]) == [[4, 4], [4, 4]]
# assert exercise_max_filter([[5, 0, 0, 1]]) == [[5, 5, 1, 1]]

**Bài 5 — TODO:** làm mịn với bán kính `k` bất kỳ (khối `(2k+1)×(2k+1)`), dùng prefix 2D.

In [25]:
def exercise_smoother_k(img: list[list[int]], k: int) -> list[list[int]]:
    # TODO: dựng prefix 2D; với mỗi ô kẹp r1,r2,c1,c2 theo k; total // count
    pass

# assert exercise_smoother_k([[100, 200, 100], [200, 50, 200], [100, 200, 100]], 1) == [[137, 141, 137], [141, 138, 141], [137, 141, 137]]
# assert exercise_smoother_k([[1, 2, 3], [4, 5, 6], [7, 8, 9]], 5) == [[5, 5, 5], [5, 5, 5], [5, 5, 5]]

**Bài 6 — Mini challenge:** áp dụng bộ làm mịn `t` lần liên tiếp. Sau bao nhiêu lần thì ảnh `[[0, 255], [255, 0]]` trở thành hằng số? Viết hàm trả về số lần đó (giới hạn tối đa 100 lần, nếu không ổn định thì trả `-1`).

In [26]:
def exercise_steps_to_constant(img: list[list[int]]) -> int:
    # TODO: lặp: nếu mọi ô bằng nhau -> trả số bước; ngược lại img = Solution().imageSmoother(img)
    pass

# assert exercise_steps_to_constant([[0, 255], [255, 0]]) == 1
# assert exercise_steps_to_constant([[3, 3], [3, 3]]) == 0

## 15. Gợi ý 3 mức

| Bài | Mức 1 | Mức 2 | Mức 3 |
|---|---|---|---|
| 1 | Ảnh 2×2: mọi ô là góc | Mỗi ô thấy cả 4 ô | `20 // 4` |
| 2 | Góc luôn 4 ô | Cạnh không kể góc: `2(m-2) + 2(n-2)` | Trong: `(m-2)(n-2)` |
| 3 | Bỏ phần cộng `total` | Chỉ tăng `count` | `count = (r2-r1+1)*(c2-c1+1)` |
| 4 | Khởi tạo `best = -1` | `best = max(best, img[nr][nc])` | Dùng `range` đã kẹp biên |
| 5 | Copy `SolutionPrefix` | Thay `1` bằng `k` khi kẹp | `min(m - 1, r + k)` |
| 6 | Hàm kiểm tra hằng số | `len({v for row in img for v in row}) == 1` | vòng `for step in range(101)` |

## 16. Đáp án tham khảo

**Bài 1:** tổng = 20, mỗi ô có 4 ô hợp lệ → `[[5, 5], [5, 5]]`.

**Bài 2:** ảnh 4×5: `count=4` có **4** ô (góc); `count=6` có `2·(4−2) + 2·(5−2) = 4 + 6 = 10` ô; `count=9` có `(4−2)·(5−2) = 6` ô. Tổng `4 + 10 + 6 = 20 = 4·5` ✔.

In [27]:
print("Bài 1:", Solution().imageSmoother([[2, 4], [6, 8]]))
from collections import Counter

def answer_count_matrix(m, n):
    return [[(min(m - 1, r + 1) - max(0, r - 1) + 1) * (min(n - 1, c + 1) - max(0, c - 1) + 1)
             for c in range(n)] for r in range(m)]

print("Bài 2:", Counter(v for row in answer_count_matrix(4, 5) for v in row))

Bài 1: [[5, 5], [5, 5]]
Bài 2: Counter({6: 10, 9: 6, 4: 4})


In [28]:
def answer_max_filter(img):
    m, n = len(img), len(img[0])
    res = [[0] * n for _ in range(m)]
    for r in range(m):
        for c in range(n):
            best = -1
            for nr in range(max(0, r - 1), min(m, r + 2)):
                for nc in range(max(0, c - 1), min(n, c + 2)):
                    best = max(best, img[nr][nc])
            res[r][c] = best
    return res


def answer_smoother_k(img, k):
    m, n = len(img), len(img[0])
    P = build_prefix_2d(img)
    res = [[0] * n for _ in range(m)]
    for r in range(m):
        r1, r2 = max(0, r - k), min(m - 1, r + k)
        for c in range(n):
            c1, c2 = max(0, c - k), min(n - 1, c + k)
            total = P[r2 + 1][c2 + 1] - P[r1][c2 + 1] - P[r2 + 1][c1] + P[r1][c1]
            res[r][c] = total // ((r2 - r1 + 1) * (c2 - c1 + 1))
    return res


def answer_steps_to_constant(img):
    for step in range(101):
        if len({v for row in img for v in row}) == 1:
            return step
        img = Solution().imageSmoother(img)
    return -1


assert answer_count_matrix(3, 3) == [[4, 6, 4], [6, 9, 6], [4, 6, 4]]
assert answer_count_matrix(1, 1) == [[1]]
assert answer_max_filter([[1, 2], [3, 4]]) == [[4, 4], [4, 4]]
assert answer_max_filter([[5, 0, 0, 1]]) == [[5, 5, 1, 1]]
assert answer_smoother_k([[100, 200, 100], [200, 50, 200], [100, 200, 100]], 1) == [[137, 141, 137], [141, 138, 141], [137, 141, 137]]
assert answer_smoother_k([[1, 2, 3], [4, 5, 6], [7, 8, 9]], 5) == [[5, 5, 5], [5, 5, 5], [5, 5, 5]]
assert answer_steps_to_constant([[0, 255], [255, 0]]) == 1
assert answer_steps_to_constant([[3, 3], [3, 3]]) == 0
for _ in range(100):
    m, n = rng.randint(1, 6), rng.randint(1, 6)
    img = [[rng.randint(0, 255) for _ in range(n)] for _ in range(m)]
    assert answer_smoother_k(img, 1) == Solution().imageSmoother(img)
print("Đáp án tham khảo: tất cả assert PASS")

Đáp án tham khảo: tất cả assert PASS


## 17. Template nhớ nhanh + checklist submit

```python
class Solution:
    def imageSmoother(self, img: list[list[int]]) -> list[list[int]]:
        m, n = len(img), len(img[0])
        res = [[0] * n for _ in range(m)]          # ma trận kết quả riêng
        for r in range(m):
            for c in range(n):
                total = count = 0
                for nr in range(max(0, r - 1), min(m, r + 2)):
                    for nc in range(max(0, c - 1), min(n, c + 2)):
                        total += img[nr][nc]
                        count += 1
                res[r][c] = total // count
        return res
```

**Câu thần chú:** *"Đọc ảnh cũ, ghi ảnh mới; kẹp biên bằng max/min; chia cho số ô thật."*

**Checklist:**
- [ ] Không ghi đè `img` khi còn cần đọc giá trị gốc.
- [ ] Tạo `res` bằng list comprehension, không dùng `[[0]*n]*m`.
- [ ] Kiểm tra / kẹp biên cả 4 phía; nhớ chỉ số âm không báo lỗi.
- [ ] Chia cho `count` thật, dùng `//`.
- [ ] Test ảnh `1×1`, `1×n`, `m×1`.